# Train Le-ADMM on Colab

Le-ADMM is the repo's ADMM solver unrolled for 5 iterations with learned τ, μ₁, μ₂, μ₃ per iteration, plus a U-Net denoiser in the `le_admm.yaml` variant. Before running, choose **Runtime > Change runtime type > GPU**.

Run folders and checkpoints are saved to Google Drive, so they survive a disconnect.

## 1. Check the GPU and mount Drive

In [ ]:
!nvidia-smi -L

from google.colab import drive
drive.mount("/content/drive")

EXPERIMENTS_ROOT = "/content/drive/MyDrive/ece488/experiments"
CHECKPOINT_ROOT = "/content/drive/MyDrive/ece488/checkpoints"
!mkdir -p {EXPERIMENTS_ROOT} {CHECKPOINT_ROOT}

## 2. Get the code

In [ ]:
import os
from pathlib import Path

BRANCH = "main"
REPO_DIR = Path("/content/ECE488")

if not REPO_DIR.exists():
    !git clone --quiet --branch {BRANCH} https://github.com/olivcho/ECE488.git {REPO_DIR}

os.chdir(REPO_DIR)
!git pull --quiet --ff-only
!git log -1 --format='%h %s'
!git status --short

## 3. Install dependencies

This keeps Colab's own PyTorch so it matches the GPU drivers. Each run records the versions it used.

In [ ]:
%pip install -q lpips torchmetrics datasets huggingface_hub pyyaml pytest tifffile
%pip install -q --no-deps -e .

# A first local download stalled on Hugging Face's Xet path, so use plain HTTPS.
os.environ["HF_HUB_DISABLE_XET"] = "1"

!python -m pytest -q

## 4. Short test

This trains Le-ADMM-U on 500 images for 2 epochs. Check that the previews look sensible and that memory fits, since backpropagation keeps every unrolled iteration's full-plane variables. Then estimate the training run:

`training epoch seconds ≈ test epoch seconds × 5,000 / 500`

If memory runs out, lower `train.batch_size` in the config.

In [ ]:
!python scripts/train_le_admm.py --config configs/le_admm_colab_test.yaml --experiments-root {EXPERIMENTS_ROOT} --checkpoint-root {CHECKPOINT_ROOT}

## 5. Training runs

Both use the same 5,000 training rows and 1,000 validation images as `unet_small.yaml`, so the three models are directly comparable. The plain variant learns only 21 numbers, so it is the cleanest test of transfer by PSF swap.

In [ ]:
!python scripts/train_le_admm.py --config configs/le_admm_plain.yaml --experiments-root {EXPERIMENTS_ROOT} --checkpoint-root {CHECKPOINT_ROOT}

In [ ]:
!python scripts/train_le_admm.py --config configs/le_admm.yaml --experiments-root {EXPERIMENTS_ROOT} --checkpoint-root {CHECKPOINT_ROOT}

## 6. Resume after a disconnect

Run sections 1 to 3 again, then set `RUN` to the folder printed by the training run.

In [ ]:
RUN = f"{EXPERIMENTS_ROOT}/le_admm_unet/YYMMDD_le_admm_v1"  # Replace with your run folder
!python scripts/train_le_admm.py --resume {RUN} --checkpoint-root {CHECKPOINT_ROOT}

## 7. See the results and the learned penalties

In [ ]:
import pandas as pd
import torch
from IPython.display import Image, display

RUN = f"{EXPERIMENTS_ROOT}/le_admm_plain/YYMMDD_le_admm_v1"  # Replace with your run folder
display(pd.read_csv(f"{RUN}/metrics.csv"))
display(Image(str(sorted(Path(RUN, "previews").glob("*.png"))[-1])))

weights = torch.load(f"{CHECKPOINT_ROOT}/{Path(RUN).parent.name}/{Path(RUN).name}/best.pt", map_location="cpu", weights_only=False)["model"]
learned = pd.DataFrame(weights["log_mu"].exp().numpy(), columns=["mu1", "mu2", "mu3"])
learned.insert(0, "tau", weights["log_tau"].exp().numpy())
learned.index.name = "iteration"
print(f"gain: {weights['log_gain'].exp():.3f}")
display(learned)

## 8. Test split and PSF swap

`evaluate_model.py` rebuilds the network from the checkpoint and scores four conditions on one split: the real measurements, measurements simulated with the training PSF, measurements simulated with a second PSF while the network keeps the old one, and the same measurements after the second PSF is swapped in. The second camera is the training PSF rotated by 180°, since the dataset has only one real PSF. Run on `val` while deciding anything, and on `test` once for the report.

In [ ]:
RUN = f"{EXPERIMENTS_ROOT}/le_admm_plain/YYMMDD_le_admm_v1"  # Replace with your run folder
CHECKPOINT = f"{CHECKPOINT_ROOT}/{Path(RUN).parent.name}/{Path(RUN).name}/best.pt"
!python scripts/evaluate_model.py --config configs/evaluate.yaml --checkpoint {CHECKPOINT} --experiments-root {EXPERIMENTS_ROOT} --set data.split=val
!python scripts/evaluate_model.py --config configs/evaluate.yaml --checkpoint {CHECKPOINT} --experiments-root {EXPERIMENTS_ROOT} --set data.split=test

## 9. Compare with the U-Net

The same script scores a U-Net checkpoint. It has no PSF to swap, so its `new_psf_no_swap` row is the whole story for that model.

In [ ]:
UNET_RUN = f"{EXPERIMENTS_ROOT}/unet_small/YYMMDD_unet_v1"  # Replace with the U-Net run folder
UNET_CHECKPOINT = f"{CHECKPOINT_ROOT}/{Path(UNET_RUN).parent.name}/{Path(UNET_RUN).name}/best.pt"
!python scripts/evaluate_model.py --config configs/evaluate.yaml --checkpoint {UNET_CHECKPOINT} --experiments-root {EXPERIMENTS_ROOT} --set data.split=test

import glob
latest = sorted(glob.glob(f"{EXPERIMENTS_ROOT}/evaluate/*/metrics.csv"))[-3:]
display(pd.concat([pd.read_csv(path).assign(run=Path(path).parent.name) for path in latest]))